# 4. Chunking for retrieval

Generic chunkers split by size and can cut an Assessment away from its Plan.
`SectionAwareChunker` never crosses a section boundary. This notebook shows the
difference on one note.

> Every note, patient, number and identifier in this notebook is **fictitious**. It runs offline, downloads no model, and uses no real patient data.

In [1]:
import os

# Keep OpenBTK's routine debug lines out of this notebook's output.
os.environ.setdefault("OPENBTK_LOG_LEVEL", "warning")

'warning'

In [2]:
from openbtk.data.clinical_text.chunking import FixedTokenChunker, SectionAwareChunker
from openbtk.data.clinical_text.preprocessing import SectionSegmenter
from openbtk.data.clinical_text.schemas import ClinicalTextRecord

NOTE = (
    "Chief Complaint:\n"
    "Shortness of breath on exertion for three weeks.\n"
    "Assessment:\n"
    "Heart failure with reduced ejection fraction, decompensated. Volume overloaded.\n"
    "Plan:\n"
    "Increase furosemide to 40 mg daily. Repeat echocardiogram in six weeks.\n"
)
record = ClinicalTextRecord(record_id="note-1", source="synthetic", text=NOTE)
record = SectionSegmenter().process(record)
list(record.sections)

['chief_complaint', 'assessment', 'plan']

## Fixed-size chunks ignore structure

With a small token budget the fixed chunker cuts wherever the budget runs out.

In [3]:
fixed = list(FixedTokenChunker(max_tokens=14).chunk(record))
for c in fixed:
    print(f"[{c.token_count:2} tok] {c.text.strip()!r}")

[14 tok] 'Chief Complaint:\nShortness of breath on exertion for three weeks.\nAssessment:\nHeart failure with'
[14 tok] 'reduced ejection fraction, decompensated. Volume overloaded.\nPlan:\nIncrease furosemide to 40 mg daily. Repeat'
[ 4 tok] 'echocardiogram in six weeks.'


Some chunks begin mid-sentence or hold the tail of the Assessment together with the
head of the Plan - so a retriever can return a chunk whose treatment has no
diagnosis attached.

## Section-aware chunks respect them

In [4]:
aware = list(SectionAwareChunker(max_tokens=14).chunk(record))
for c in aware:
    print(f"[{c.section:>16}] [{c.token_count:2} tok] {c.text.strip()!r}")

# No chunk spans two sections; long sections are split *inside* themselves.
for c in aware:
    inside = [
        name
        for name, span in record.sections.items()
        if span.start <= c.span.start and c.span.end <= span.end
    ]
    assert len(inside) == 1, (c.section, inside)

[ chief_complaint] [ 8 tok] 'Shortness of breath on exertion for three weeks.'
[      assessment] [ 9 tok] 'Heart failure with reduced ejection fraction, decompensated. Volume overloaded.'
[            plan] [11 tok] 'Increase furosemide to 40 mg daily. Repeat echocardiogram in six weeks.'


## Token counts are approximate by default

`count_tokens_approximate` is a labelled heuristic. If a model's context window is
what you are budgeting, pass a real tokenizer's counter:

```python
SectionAwareChunker(max_tokens=512, count_tokens=my_tokenizer_count)
```

Each chunk records its `span` in the parent record, so a retrieved chunk can
always be traced to exact characters in the source.

In [5]:
c = aware[0]
print(c.chunk_id, c.record_id, (c.span.start, c.span.end))
assert NOTE[c.span.start : c.span.end] == c.text

note-1:0 note-1 (17, 66)
